# Masked Language Models (MLM): BERT and DistilBERT

## Overview
BERT reads text in both directions at once. It is pre-trained by hiding words and predicting them, then reused for many tasks by attaching a small task-specific head.

In this notebook you will:
1. **See how BERT learns**: predict masked words, inspect token probabilities, and show that BERT uses context on *both* sides of the mask.
2. **See contextual embeddings**: the same word gets different vectors in different sentences.
3. **Audit bias**: compare fill-in-the-blank predictions for men and women.
4. **Understand task heads**: why an untrained head gives random output, and what a fine-tuned head does (sentiment, NER).
5. **Compare BERT and DistilBERT**: parameters, speed, and agreement.
6. **Fine-tune DistilBERT on IMDB** and measure accuracy before and after.

In [1]:
# The Hugging Face Trainer needs `accelerate`. Install once, then restart the kernel if it was missing.
%pip install -q "accelerate>=1.1.0" datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModel, AutoModelForMaskedLM,
                          AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, pipeline)

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: mps


---
## 1. How BERT Learns: Masked Language Modeling

During pre-training, about **15% of the tokens** are chosen. Of those, 80% are replaced by `[MASK]`, 10% by a random token, and 10% are left unchanged. The model must predict the original token at every chosen position, using the whole sentence.

Below we load BERT with its masked-language-modeling head and write a small helper that shows the **top predictions with their probabilities** (a softmax over the 30,522-word vocabulary at the `[MASK]` position).

In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
mlm = AutoModelForMaskedLM.from_pretrained("bert-base-uncased").eval()

def top_k(text, k=5, model=mlm, tok=tokenizer):
    """Top-k predictions (token, probability) for the single [MASK] in `text`."""
    enc = tok(text, return_tensors="pt")
    pos = (enc["input_ids"][0] == tok.mask_token_id).nonzero().item()
    with torch.no_grad():
        logits = model(**enc).logits[0, pos]
    probs = torch.softmax(logits, dim=-1)
    top = probs.topk(k)
    return [(tok.decode(int(i)), round(float(p), 3)) for p, i in zip(top.values, top.indices)]

print("Vocabulary size:", tokenizer.vocab_size)
for token, p in top_k("The capital of France is [MASK]."):
    print(f"{token:>10}  {p:.3f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vocabulary size: 30522
     paris  0.417
     lille  0.071
      lyon  0.063
 marseille  0.044
     tours  0.030


In [4]:
# Several answers can be valid: the model spreads probability over them
for token, p in top_k("Hello, I'm a [MASK] model."):
    print(f"{token:>10}  {p:.3f}")

      role  0.144
   fashion  0.142
       new  0.062
     super  0.041
  business  0.026


### 1.1 Bidirectional context

A left-to-right model (like GPT) only sees the words *before* the blank. BERT also sees the words *after* it. We hide the right-hand context and compare.

In [5]:
pairs = [
    ("I went to the [MASK] to deposit my paycheck.", "I went to the [MASK]."),
    ("The [MASK] barked at the mailman and chased him down the street.", "The [MASK]."),
]
rows = []
for full, left_only in pairs:
    rows.append({"Context": "full sentence", "Text": full, "Top-5": [t for t, _ in top_k(full)]})
    rows.append({"Context": "left only",     "Text": left_only, "Top-5": [t for t, _ in top_k(left_only)]})
pd.DataFrame(rows)

,Context,Text,Top-5
0,full sentence,I went to the [MASK] to deposit my paycheck.,"[bank, office, store, teller, atm]"
1,left only,I went to the [MASK].,"[bathroom, door, kitchen, window, closet]"
2,full sentence,The [MASK] barked at the mailman and chased him down the street.,"[dog, man, dogs, boy, driver]"
3,left only,The [MASK].,"[guardian, independent, times, voice, sun]"


With only the left context the predictions are generic; once the words *after* the blank are visible, the model picks the right word. This is what "bidirectional" means, and it is why BERT is strong for understanding tasks but cannot generate text left to right.

### 1.2 Contextual embeddings

Older embeddings (word2vec, GloVe) give a word **one fixed vector**. BERT's vector for a word depends on the sentence. We compare the vector of `bank` in three sentences, and also the static lookup-table vector that BERT starts from.

In [6]:
bert = AutoModel.from_pretrained("bert-base-uncased").eval()

sentences = {
    "river":   "She sat on the bank of the river and watched the water.",
    "money 1": "He deposited his paycheck at the bank on Monday.",
    "money 2": "The bank approved her loan application.",
}

def word_vector(sentence, word="bank"):
    enc = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    with torch.no_grad():
        hidden = bert(**enc).last_hidden_state[0]
    return hidden[tokens.index(word)]

vecs = {name: word_vector(s) for name, s in sentences.items()}
static = bert.embeddings.word_embeddings.weight[tokenizer.convert_tokens_to_ids("bank")].detach()

cos = torch.nn.functional.cosine_similarity
names = list(vecs)
table = pd.DataFrame([[cos(vecs[a], vecs[b], dim=0).item() for b in names] for a in names],
                     index=names, columns=names).round(2)
print("Cosine similarity between the contextual vectors of 'bank':")
display(table)
print("Static (input-embedding) vector for 'bank' is identical in every sentence.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cosine similarity between the contextual vectors of 'bank':


,river,money 1,money 2
river,1.00,0.48,0.42
money 1,0.48,1.00,0.80
money 2,0.42,0.80,1.00


Static (input-embedding) vector for 'bank' is identical in every sentence.


Expect the two *money* sentences to be more similar to each other than either is to the *river* sentence: BERT has encoded the meaning of `bank` in context.

### 1.3 Bias in the training data

BERT is trained on Wikipedia and BookCorpus, and it learns the associations in that text. We compare the predictions for "man" and "woman" side by side with their probabilities.

In [7]:
def bias_table(template, groups=("man", "woman"), k=5):
    cols = {}
    for g in groups:
        preds = top_k(template.format(g), k=k)
        cols[g] = [f"{t} ({p:.2f})" for t, p in preds]
    return pd.DataFrame(cols, index=[f"#{i+1}" for i in range(k)])

display(bias_table("This {} works as a [MASK]."))
display(bias_table("The {} was very [MASK].", groups=("boy", "girl")))

,man,woman
#1,carpenter (0.07),nurse (0.13)
#2,lawyer (0.05),maid (0.07)
#3,farmer (0.04),teacher (0.07)
#4,businessman (0.03),waitress (0.06)
#5,doctor (0.03),prostitute (0.04)


,boy,girl
#1,young (0.11),pretty (0.17)
#2,quiet (0.05),beautiful (0.10)
#3,small (0.04),young (0.06)
#4,handsome (0.02),attractive (0.03)
#5,pretty (0.02),quiet (0.02)


Look at the two columns: the occupations and adjectives differ systematically by gender. The model did not "decide" this; it reflects its training text. Any application built on a pre-trained encoder inherits these associations unless they are measured and mitigated.

**Discussion:** try other templates (religion, nationality, age). Which differences would be harmful in a hiring or screening tool?

---
## 2. Task Heads: From Pre-Trained Body to Useful Model

For classification, NER or QA, we keep the pre-trained BERT body and add a small **task head** (a linear layer). The head starts with **random weights**, so before fine-tuning its output is meaningless. The loading messages ("MISSING ... newly initialized") are the warning.

In [8]:
test_sentences = ["I loved this movie, it was fantastic!", "This was a terrible, boring film."]

# 1) Pre-trained body + RANDOM classification head
raw = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).eval()
with torch.no_grad():
    enc = tokenizer(test_sentences, return_tensors="pt", padding=True)
    raw_probs = torch.softmax(raw(**enc).logits, dim=-1)

# 2) The same kind of model, but fine-tuned on sentiment (SST-2)
tuned = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
tuned_out = tuned(test_sentences)

pd.DataFrame({
    "Sentence": test_sentences,
    "Untrained head P(positive)": [round(float(p), 2) for p in raw_probs[:, 1]],
    "Fine-tuned head": [f"{o['label']} ({o['score']:.2f})" for o in tuned_out],
})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,Sentence,Untrained head P(positive),Fine-tuned head
0,"I loved this movie, it was fantastic!",0.4,POSITIVE (1.00)
1,"This was a terrible, boring film.",0.4,NEGATIVE (1.00)


The untrained head gives nearly the same answer for both reviews (close to 50/50 or arbitrary). The fine-tuned model separates them with high confidence. **The body is shared; the head and the fine-tuning are what create the skill.**

### 2.1 Named Entity Recognition (NER)

NER puts a classification head on *every token*. We use a BERT model already fine-tuned for NER (`dslim/bert-base-NER`) and group sub-word pieces into entities.

In [9]:
ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple")
text = "Tim Cook announced that Apple will open a new office in Austin, Texas, next to the University of Texas."
pd.DataFrame(ner(text))[["word", "entity_group", "score"]].round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,word,entity_group,score
0,Tim Cook,PER,1.000
1,Apple,ORG,0.999
2,Austin,LOC,0.999
3,Texas,LOC,0.999
4,University of Texas,ORG,0.999


`PER` = person, `ORG` = organization, `LOC` = location, `MISC` = other. The same pre-trained body with a different head and different labels gives a different task. Question answering (notebook 3.4) uses two outputs per token: start and end of the answer span.

---
## 3. DistilBERT: A Smaller, Faster BERT

DistilBERT is trained by **knowledge distillation**: a small student (6 layers) learns to mimic the larger BERT teacher (12 layers). We check the claims instead of trusting them: number of parameters, speed, and how often the two models agree.

In [10]:
distil_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
distil = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased").eval()

def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6

def time_model(model, tok, text, batch=32, repeats=5):
    enc = tok([text] * batch, return_tensors="pt", padding=True)
    times = []
    with torch.no_grad():
        model(**enc)  # warm-up
        for _ in range(repeats):
            t0 = time.perf_counter(); model(**enc); times.append(time.perf_counter() - t0)
    return float(np.median(times)) * 1000

text = "The quick brown fox jumps over the lazy dog while the sun sets slowly behind the hills."
t_bert, t_distil = time_model(mlm, tokenizer, text), time_model(distil, distil_tok, text)

masked = ["The capital of France is [MASK].", "She poured the coffee into a [MASK].",
          "The doctor told him to take the [MASK] twice a day.", "Water boils at one hundred degrees [MASK].",
          "He plays the [MASK] in a rock band."]
agree = sum(top_k(s)[0][0] == top_k(s, model=distil, tok=distil_tok)[0][0] for s in masked)

pd.DataFrame({
    "Model": ["BERT-base", "DistilBERT"],
    "Parameters (M)": [round(n_params(mlm), 1), round(n_params(distil), 1)],
    "Batch of 32 (ms, CPU)": [round(t_bert), round(t_distil)],
}).assign(**{"Relative speed": ["1.0x", f"{t_bert / t_distil:.1f}x"]})

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,Model,Parameters (M),"Batch of 32 (ms, CPU)",Relative speed
0,BERT-base,109.5,181,1.0x
1,DistilBERT,67.0,122,1.5x


In [11]:
print(f"Top-1 predictions identical on {agree}/{len(masked)} masked sentences\n")
for s in masked:
    print(f"{s}\n   BERT: {top_k(s)[0][0]:<12} DistilBERT: {top_k(s, model=distil, tok=distil_tok)[0][0]}")

Top-1 predictions identical on 3/5 masked sentences

The capital of France is [MASK].
   BERT: paris        DistilBERT: marseille
She poured the coffee into a [MASK].
   BERT: mug          DistilBERT: mug
The doctor told him to take the [MASK] twice a day.
   BERT: pill         DistilBERT: medication
Water boils at one hundred degrees [MASK].
   BERT: temperature  DistilBERT: temperature
He plays the [MASK] in a rock band.
   BERT: guitar       DistilBERT: guitar


DistilBERT has roughly 40% fewer parameters and runs noticeably faster while usually predicting the same words. Distillation is covered again in Week 5.

---
## 4. Fine-Tuning DistilBERT on IMDB

We add a classification head to DistilBERT and fine-tune on a **small subset** of IMDB movie reviews (1,000 train / 500 test). We measure accuracy **before** training (random head, about 50%) and **after**.

In [12]:
dataset = load_dataset("imdb")
ft_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return ft_tok(batch["text"], truncation=True, max_length=256)

train_ds = dataset["train"].shuffle(seed=42).select(range(1000)).map(tokenize, batched=True, remove_columns=["text"])
test_ds  = dataset["test"].shuffle(seed=42).select(range(500)).map(tokenize, batched=True, remove_columns=["text"])

clf = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": float((np.argmax(logits, axis=-1) == labels).mean())}

args = TrainingArguments(
    output_dir="./results_imdb", num_train_epochs=1, learning_rate=5e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="no", save_strategy="no", logging_steps=20, report_to="none",
)
trainer = Trainer(model=clf, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                  processing_class=ft_tok, data_collator=DataCollatorWithPadding(ft_tok),
                  compute_metrics=compute_metrics)

before = trainer.evaluate()["eval_accuracy"]
print(f"Accuracy BEFORE fine-tuning: {before:.3f}")

Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at /Users/apujari/.cache/huggingface/datasets/imdb/plain_text/0.0.0/e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Tue Oct 13 16:13:58 2026).


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/homebrew/anaconda3/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__in

Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Impacting Interactivity (0000000e:kIOGPUCommandBufferCallbackErrorImpactingInteractivity)
	<AGXG14XFamilyCommandBuffer: 0x154ce7360>
    label = <none> 
    device = <AGXG14CDevice: 0x153a69c00>
        name = Apple M2 Max 
    commandQueue = <AGXG14XFamilyCommandQueue: 0x127da0000>
        label = <none> 
        device = <AGXG14CDevice: 0x153a69c00>
            name = Apple M2 Max 
    retainedReferences = 1


Training Loss,Validation Loss,Step,Accuracy
No log,0.689873,0,0.570000


Accuracy BEFORE fine-tuning: 0.570


In [13]:
trainer.train()
after = trainer.evaluate()["eval_accuracy"]
print(f"\nAccuracy BEFORE fine-tuning: {before:.3f}")
print(f"Accuracy AFTER  fine-tuning: {after:.3f}")

Step,Training Loss
20,0.641773
40,0.448356
60,0.355756


Training Loss,Validation Loss,Step,Accuracy
0.355756,0.341155,63,0.854000



Accuracy BEFORE fine-tuning: 0.570
Accuracy AFTER  fine-tuning: 0.854


In [14]:
# Look at a few predictions from the fine-tuned model
raw_test = dataset["test"].shuffle(seed=42).select(range(500))
rows = []
clf_model = trainer.model.eval()
for i in range(6):
    review, gold = raw_test[i]["text"], raw_test[i]["label"]
    enc = ft_tok(review, truncation=True, max_length=256, return_tensors="pt").to(clf_model.device)
    with torch.no_grad():
        p = torch.softmax(clf_model(**enc).logits, dim=-1)[0]
    rows.append({"Review": review[:90] + "...", "Gold": ["neg", "pos"][gold],
                 "Predicted": ["neg", "pos"][int(p.argmax())], "Confidence": round(float(p.max()), 2)})
pd.DataFrame(rows)

,Review,Gold,Predicted,Confidence
0,"<br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in...",pos,pos,0.91
1,"This is the latest entry in the long series of films with the French agent, ...",pos,pos,0.85
2,This movie was so frustrating. Everything seemed energetic and I was totally...,neg,neg,0.94
3,"I was truly and wonderfully surprised at ""O' Brother, Where Art Thou?"" The v...",pos,pos,0.64
4,This movie spends most of its time preaching that it is the script that make...,neg,neg,0.65
5,After a very long time Marathi cinema has come with some good movie.This mov...,pos,pos,0.93


## 5. Summary & Key Takeaways

1. **Masked Language Modeling**: BERT hides about 15% of tokens and predicts them from context on *both* sides, which gives it deep contextual representations.
2. **Contextual embeddings**: the same word (`bank`) gets different vectors in different sentences; static embeddings cannot do this.
3. **Bias is inherited**: fill-mask outputs differ systematically by gender because the training text does. Always audit.
4. **Task heads**: a pre-trained body plus a randomly initialized head gives random output; fine-tuning (or a model someone already fine-tuned) is what makes it useful.
5. **Distillation**: DistilBERT keeps most of BERT's behavior with about 40% fewer parameters and faster inference.
6. **Fine-tuning with little data works**: a few hundred labeled reviews move accuracy from chance to well above it (measure it, as above).